
# Parish maps of avoided EADs from coastal-flood mangrove protection and river-flood forest restoration

This notebook creates stacked parish-map panels for the spatial service-provision comparison in paper 3.

It uses the cross-hazard parish summary tables generated by `spatial_comparison_coastal_mangroves_river_restoration.ipynb`:

- coastal flooding: net mangrove-attributed avoided EAD by parish;
- river flooding: forest-restoration avoided EAD by parish from Haggis et al.

The main outputs are stacked two-panel maps, with coastal flooding above river flooding. Absolute-value maps are shown in US$ million, and percentage-share maps show each parish's share of the relevant national avoided EAD total. Maximum-scenario outputs are intended for the main text; minimum-scenario outputs are generated as supplementary equivalents.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPERS = ROOT / "dphil_papers"
ROBYN_LIBRARIES = PAPERS / "robyns_libraries"
if str(ROBYN_LIBRARIES) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARIES))

import Robyn_paper_2_defs

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
PAPER3 = PAPERS / "dphil_paper_3"
COMMON = PAPERS / "dphil_common_cross_cutting"

SPATIAL_DIR = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison"
OUT_DIR = SPATIAL_DIR / "parish_maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARISH_PATH = COMMON / "common_incoming_data" / "boundaries" / "jam_adm_shp" / "jam_admbnda_adm1.shp"
COASTAL_PARISH_PATH = SPATIAL_DIR / "coastal_flood_mangrove_avoided_ead_by_parish_minmax.csv"
RIVER_PARISH_PATH = SPATIAL_DIR / "river_flood_restoration_avoided_ead_by_parish_minmax.csv"

for path in [PARISH_PATH, COASTAL_PARISH_PATH, RIVER_PARISH_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

CRS_METRIC = "EPSG:3448"
OUT_DIR


## Load and merge parish results

In [ ]:
def display_parish_name(name: str) -> str:
    replacements = {
        "Saint Andrew": "St Andrew",
        "Saint Ann": "St Ann",
        "Saint Catherine": "St Catherine",
        "Saint Elizabeth": "St Elizabeth",
        "Saint James": "St James",
        "Saint Mary": "St Mary",
        "Saint Thomas": "St Thomas",
    }
    return replacements.get(name, name)


parishes = gpd.read_file(PARISH_PATH).to_crs(CRS_METRIC)[["NAME_1", "geometry"]].copy()
parishes["parish"] = parishes["NAME_1"].map(display_parish_name)

coastal = pd.read_csv(COASTAL_PARISH_PATH)
river = pd.read_csv(RIVER_PARISH_PATH)

# Use full parish names for joining to the boundary file.
coastal_join = coastal.rename(columns={"parish_raw": "NAME_1"})
river_join = river.copy()

coastal_map = parishes.merge(coastal_join, on=["NAME_1", "parish"], how="left")
river_map = parishes.merge(river_join, on=["NAME_1", "parish"], how="left")

for parish_map in [coastal_map, river_map]:
    for column_name in parish_map.columns:
        if column_name.endswith("_minimum") or column_name.endswith("_maximum"):
            if pd.api.types.is_numeric_dtype(parish_map[column_name]):
                parish_map[column_name] = parish_map[column_name].fillna(0.0)

coastal_unassigned = coastal.loc[coastal["parish"].eq("Unassigned")].copy()

# The parish boundary file includes small offshore cays. Use the main-island
# polygon parts for map extents so the panels are not dominated by ocean whitespace.
parish_parts = parishes.explode(index_parts=False).copy()
main_island_parts = parish_parts.loc[parish_parts.geometry.area > 1_000_000].copy()
MAIN_ISLAND_BOUNDS = main_island_parts.total_bounds

# Label offsets are in metres in EPSG:3448 and keep crowded south-east parish names readable.
PARISH_LABEL_OFFSETS_M = {
    "Clarendon": (-2000, -3500),
    "Kingston": (10500, -6500),
    "Manchester": (-1000, 3500),
    "St Andrew": (5000, 9500),
    "St Catherine": (-5500, -2500),
    "St Thomas": (2500, -2500),
}
PARISH_LABEL_LEADER_LINES = {"Kingston"}

coastal_unassigned[["avoided_ead_usd_minimum", "avoided_ead_usd_maximum", "share_of_assigned_total_pct_minimum", "share_of_assigned_total_pct_maximum"]]


In [ ]:

def add_scenario_columns(gdf: gpd.GeoDataFrame, value_prefix: str, scenario: str) -> gpd.GeoDataFrame:
    out = gdf.copy()
    value_col = f"{value_prefix}_{scenario}"
    total = float(out[value_col].sum())
    out[f"{value_prefix}_million_{scenario}"] = out[value_col] / 1e6
    out[f"share_pct_mapped_{scenario}"] = np.where(total > 0, out[value_col] / total * 100, 0.0)
    return out


for scenario in ["minimum", "maximum"]:
    coastal_map = add_scenario_columns(coastal_map, "avoided_ead_usd", scenario)
    river_map = add_scenario_columns(river_map, "avoided_ead_usd", scenario)

summary_rows = []
for scenario in ["minimum", "maximum"]:
    for hazard, gdf in [("coastal_flood", coastal_map), ("river_flood", river_map)]:
        value_col = f"avoided_ead_usd_{scenario}"
        top = gdf[["parish", value_col]].sort_values(value_col, ascending=False).head(5).copy()
        top["scenario"] = scenario
        top["hazard"] = hazard
        top["share_pct"] = top[value_col] / gdf[value_col].sum() * 100
        summary_rows.append(top)

top5_summary = pd.concat(summary_rows, ignore_index=True)
top5_summary



## Plotting functions

The absolute-value panels use separate colour scales for the two maps because river-flood avoided EADs are much larger than coastal-flood avoided EADs. The percentage-share panels use a shared colour scale so the concentration pattern is visually comparable across hazards.


In [ ]:
NATURE_RC = {
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.titlesize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

CMAP = mpl.colormaps["YlGnBu"]
EDGE_COLOR = "#555555"
MISSING_COLOR = "#f0f0f0"


def format_scenario_label(scenario: str) -> str:
    return "Minimum scenario" if scenario == "minimum" else "Maximum scenario"


def format_money_tick(value_million: float) -> str:
    if value_million >= 10:
        return f"{value_million:.0f}"
    if value_million >= 1:
        return f"{value_million:.1f}"
    return f"{value_million:.2f}"


def choose_label_color(value: float, norm: Normalize) -> str:
    red, green, blue = CMAP(norm(value))[:3]
    relative_luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return "white" if relative_luminance < 0.46 else "#222222"


def add_parish_labels(axis, parish_geodataframe: gpd.GeoDataFrame, value_column: str, norm: Normalize) -> None:
    for _, parish_row in parish_geodataframe.iterrows():
        label_point = parish_row.geometry.representative_point()
        x_offset, y_offset = PARISH_LABEL_OFFSETS_M.get(parish_row["parish"], (0, 0))
        label_x = label_point.x + x_offset
        label_y = label_point.y + y_offset
        label_color = choose_label_color(float(parish_row[value_column]), norm)
        if parish_row["parish"] in PARISH_LABEL_LEADER_LINES:
            axis.plot(
                [label_point.x, label_x],
                [label_point.y, label_y],
                color="#555555",
                linewidth=0.35,
                zorder=8,
            )
        axis.text(
            label_x,
            label_y,
            parish_row["parish"],
            ha="center",
            va="center",
            fontsize=4.8,
            color=label_color,
            zorder=10,
        )


def add_map_furniture(axis) -> None:
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(
        axis,
        location=(0.88, 0.86),
        size=0.05,
        fontsize=8,
        label_offset=0.02,
    )


def draw_panel_map(
    scenario: str,
    value_kind: str = "absolute",
    output_stem: str | None = None,
    dpi: int = 600,
):
    """Draw a stacked coastal/river parish map panel.

    value_kind='absolute' maps avoided EAD in US$ million with separate colour scales.
    value_kind='share' maps percentage contribution with a shared colour scale.
    """
    if value_kind not in {"absolute", "share"}:
        raise ValueError("value_kind must be 'absolute' or 'share'")

    if value_kind == "absolute":
        coastal_column = f"avoided_ead_usd_million_{scenario}"
        river_column = f"avoided_ead_usd_million_{scenario}"
        coastal_label = "Avoided EAD (US$ million)"
        river_label = "Avoided EAD (US$ million)"
        coastal_norm = Normalize(vmin=0, vmax=float(coastal_map[coastal_column].max()))
        river_norm = Normalize(vmin=0, vmax=float(river_map[river_column].max()))
        norms = [coastal_norm, river_norm]
        labels = [coastal_label, river_label]
        tick_formatter = format_money_tick
    else:
        coastal_column = f"share_pct_mapped_{scenario}"
        river_column = f"share_pct_mapped_{scenario}"
        shared_max = float(max(coastal_map[coastal_column].max(), river_map[river_column].max()))
        norms = [Normalize(vmin=0, vmax=shared_max), Normalize(vmin=0, vmax=shared_max)]
        labels = ["Share of mapped avoided EAD (%)", "Share of mapped avoided EAD (%)"]
        tick_formatter = lambda value: f"{value:.0f}"

    rows = [
        ("a", "Coastal flooding: mangrove protection", coastal_map, coastal_column, norms[0], labels[0]),
        ("b", "River flooding: forest restoration", river_map, river_column, norms[1], labels[1]),
    ]

    with mpl.rc_context(NATURE_RC):
        figure, axes = plt.subplots(2, 1, figsize=(7.1, 5.65), constrained_layout=False)
        plt.subplots_adjust(left=0.02, right=0.92, top=0.95, bottom=0.05, hspace=0.10)

        for axis, (panel_letter, title, parish_geodataframe, value_column, norm, colorbar_label) in zip(axes, rows):
            axis.set_axis_off()
            axis.set_aspect("equal")
            parish_geodataframe.plot(
                ax=axis,
                column=value_column,
                cmap=CMAP,
                norm=norm,
                linewidth=0.35,
                edgecolor=EDGE_COLOR,
                missing_kwds={"color": MISSING_COLOR, "edgecolor": EDGE_COLOR, "hatch": "///"},
            )
            parish_geodataframe.boundary.plot(ax=axis, color="white", linewidth=0.65, zorder=2)
            parish_geodataframe.boundary.plot(ax=axis, color=EDGE_COLOR, linewidth=0.25, zorder=3)

            x_min, y_min, x_max, y_max = MAIN_ISLAND_BOUNDS
            x_range, y_range = x_max - x_min, y_max - y_min
            axis.set_xlim(x_min - 0.02 * x_range, x_max + 0.02 * x_range)
            axis.set_ylim(y_min - 0.03 * y_range, y_max + 0.03 * y_range)

            add_parish_labels(axis, parish_geodataframe, value_column, norm)
            add_map_furniture(axis)

            axis.text(
                0.01,
                0.98,
                panel_letter,
                transform=axis.transAxes,
                ha="left",
                va="top",
                fontsize=9,
                fontweight="bold",
            )
            axis.set_title(f"{title} ({format_scenario_label(scenario).lower()})", loc="left", pad=2)

            divider = make_axes_locatable(axis)
            colorbar_axis = divider.append_axes("right", size="2.8%", pad=0.04)
            scalar_mappable = ScalarMappable(norm=norm, cmap=CMAP)
            scalar_mappable.set_array([])
            colorbar = figure.colorbar(scalar_mappable, cax=colorbar_axis)
            colorbar.outline.set_linewidth(0.35)
            colorbar.ax.tick_params(width=0.35, length=2, labelsize=6)
            colorbar.set_label(colorbar_label, fontsize=6.5)
            ticks = colorbar.get_ticks()
            colorbar.set_ticklabels([tick_formatter(tick_value) for tick_value in ticks])

        if output_stem is None:
            output_stem = f"parish_avoided_ead_coastal_river_{scenario}_{value_kind}_panel"

        output_paths = []
        for suffix in ["png", "pdf", "svg"]:
            output_path = OUT_DIR / f"{output_stem}.{suffix}"
            save_kwargs = {"bbox_inches": "tight", "facecolor": "white"}
            if suffix == "png":
                save_kwargs["dpi"] = dpi
            figure.savefig(output_path, **save_kwargs)
            output_paths.append(output_path)
        plt.show()
        return output_paths


## Maximum scenario panel

In [ ]:

max_absolute_paths = draw_panel_map("maximum", value_kind="absolute")
max_share_paths = draw_panel_map("maximum", value_kind="share")
max_absolute_paths, max_share_paths


## Minimum scenario panel

In [ ]:

min_absolute_paths = draw_panel_map("minimum", value_kind="absolute")
min_share_paths = draw_panel_map("minimum", value_kind="share")
min_absolute_paths, min_share_paths


## Save top parish values used for checking labels and manuscript text

In [ ]:

top5_path = OUT_DIR / "parish_avoided_ead_top5_values_for_maps.csv"
top5_summary.to_csv(top5_path, index=False)
top5_summary


In [ ]:
print("Wrote figure outputs to:", OUT_DIR)
for path in sorted(OUT_DIR.glob("parish_avoided_ead_coastal_river_*_panel.*")):
    print(path.name)
print()
print("Top values:", top5_path.name)